# Bronze layer

Raw ingestion into Delta tables. No transformations here: bronze keeps the data
as close to the source as possible.

## Two ingestion paths

**Open-Meteo (historical weather)** is fetched directly from this notebook.
The host is reachable from Databricks serverless compute.

**Fingrid (electricity consumption + wind production)** is fetched by a local
script (`ingest/run_local_fetch.py`), landed as CSV in the Unity Catalog volume
`energy_weather.landing`, and read from there. `data.fingrid.fi` does not
resolve from Databricks Free Edition serverless compute, so ingestion for this
source runs outside the platform.

Separating ingestion from transformation is standard practice regardless of
this constraint: production Spark clusters are frequently network isolated,
with a dedicated ingestion layer landing data first.

## Known issues

The two paths produce different schemas for the same weather data: `time` is
`timestamp` when read from CSV, `string` when built from JSON. Column order
differs as well. Harmonised in silver, not here.

`bronze_weather` (volume path) and `bronze_weather_api` (direct path) currently
hold the same data. Kept side by side temporarily for comparison; one becomes
canonical in silver.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.energy_weather;

In [0]:
from ingest.fetch_weather import fetch_weather, load_weather_points

In [0]:
from datetime import datetime, timedelta, timezone

# 12 months of data, ending a couple of hours ago so we don't ask
# the API for data that doesn't exist yet
end_time = datetime.now(timezone.utc) - timedelta(hours=2)
start_time = end_time - timedelta(days=365)

start_str = start_time.strftime("%Y-%m-%dT%H:%M:%SZ")
end_str = end_time.strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Fetching from {start_str} to {end_str}")

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.energy_weather.landing;

In [0]:
consumption_df = spark.read.csv(
    "/Volumes/workspace/energy_weather/landing/fingrid_consumption.csv",
    header=True,
    inferSchema=True,
)
wind_df = spark.read.csv(
    "/Volumes/workspace/energy_weather/landing/fingrid_wind.csv",
    header=True,
    inferSchema=True,
)
weather_df = spark.read.csv(
    "/Volumes/workspace/energy_weather/landing/weather.csv",
    header=True,
    inferSchema=True,
)

consumption_df.write.format("delta").mode("overwrite").saveAsTable("workspace.energy_weather.bronze_consumption")
wind_df.write.format("delta").mode("overwrite").saveAsTable("workspace.energy_weather.bronze_wind")
weather_df.write.format("delta").mode("overwrite").saveAsTable("workspace.energy_weather.bronze_weather")

print("Consumption:", consumption_df.count())
print("Wind:", wind_df.count())
print("Weather:", weather_df.count())

In [0]:
%sql
SELECT * FROM workspace.energy_weather.bronze_consumption LIMIT 5;

In [0]:
import requests

# Open-Meteo archive API, no API key required.
# Different host than Fingrid: the host name is the only variable we are testing.
url = "https://archive-api.open-meteo.com/v1/archive"

# Smallest useful probe: one day, one location, one variable.
# We care about reachability, not the data itself.
params = {
    "latitude": 60.17,
    "longitude": 24.94,
    "start_date": "2025-01-01",
    "end_date": "2025-01-01",
    "hourly": "temperature_2m",
}

# timeout prevents the cell from hanging if packets are silently dropped
response = requests.get(url, params=params, timeout=10)

# Turns an HTTP error into an exception, so we can tell
# "connection worked but API complained" apart from "connection never happened"
response.raise_for_status()

payload = response.json()
print("Status:", response.status_code)
print("First 3 temperatures:", payload["hourly"]["temperature_2m"][:3])

In [0]:
# Repo root inside the Databricks workspace.
# Workspace files are readable with ordinary Python file I/O,
# so load_weather_points works here exactly as it does locally.
REPO_ROOT = "/Workspace/Users/dev@nikokurvinen.fi/energy-weather-lakehouse"
POINTS_CSV = f"{REPO_ROOT}/seeds/area_weather_points.csv"

points = load_weather_points(POINTS_CSV)

print("Points loaded:", len(points))
print("First point:", points[0])
print(list(points[0].keys()))

In [0]:
%sql
SELECT
    MIN(time) AS first_timestamp,
    MAX(time) AS last_timestamp,
    COUNT(*)  AS row_count
FROM workspace.energy_weather.bronze_weather

In [0]:
import time

# Same window as the existing bronze_weather table,
# so both ingestion paths are directly comparable.
START_DATE = "2025-09-17"
END_DATE = "2026-09-17"

rows = []

for point in points:
    # csv.DictReader returns every value as a string,
    # so coordinates must be converted explicitly.
    hourly = fetch_weather(
        latitude=float(point["latitude"]),
        longitude=float(point["longitude"]),
        start_date=START_DATE,
        end_date=END_DATE,
    )

    # Tag each row with its origin, otherwise the four
    # result sets become indistinguishable once merged.
    for row in hourly:
        row["area_id"] = point["area_id"]
        row["city"] = point["city"]

    rows.extend(hourly)
    print(f"{point['city']}: {len(hourly)} rows")

    time.sleep(1)

print("Total rows:", len(rows))

In [0]:
# Convert the Python list of dicts into a distributed Spark DataFrame.
# Schema is inferred from the dict keys.
weather_api_df = spark.createDataFrame(rows)

weather_api_df.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.energy_weather.bronze_weather_api"
)

print("Rows written:", weather_api_df.count())

In [0]:
print("bronze_weather (CSV via Volume):")
spark.table("workspace.energy_weather.bronze_weather").printSchema()

print("bronze_weather_api (direct API call):")
spark.table("workspace.energy_weather.bronze_weather_api").printSchema()